# 🍷 Wine Quality Prediction
### End-to-End Machine Learning Walkthrough

**Dataset:** UCI Wine Quality (Red + White)  
**Goal:** Predict whether a wine is *Good* (quality ≥ 7) or *Bad* (quality < 7)  
**Models:** Logistic Regression vs Random Forest Classifier

---

## 📦 0. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection     import train_test_split
from sklearn.preprocessing       import StandardScaler
from sklearn.linear_model        import LogisticRegression
from sklearn.ensemble            import RandomForestClassifier
from sklearn.metrics             import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
import joblib

# Plot style
plt.rcParams.update({
    'figure.facecolor' : '#0f0f0f',
    'axes.facecolor'   : '#1a1a1a',
    'axes.labelcolor'  : '#888880',
    'xtick.color'      : '#888880',
    'ytick.color'      : '#888880',
    'text.color'       : '#e8e0d0',
    'axes.edgecolor'   : '#2a2a2a',
    'grid.color'       : '#2a2a2a',
})

print('✅ Libraries imported successfully!')

---
## 📂 1. Load Data

The UCI Wine Quality dataset uses **semicolons** as separator (not commas).  
We handle both formats automatically.

In [ ]:
df = pd.read_csv('../data/winequality.csv', sep=None, engine='python')

print(f'Shape  : {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

---
## 🔍 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('── Basic Info ──────────────────────────────')
df.info()
print('\n── Descriptive Statistics ──────────────────')
df.describe().round(3)

In [ ]:
# Missing values check
missing = df.isnull().sum()
print('── Missing Values ──────────────────────────')
if missing.sum() == 0:
    print('✅ No missing values found!')
else:
    print(missing[missing > 0])

In [ ]:
# Wine quality distribution
# INSIGHT: Most wines score 5–6. Very few score 3 or 9.
# This imbalance is why we convert to binary (Good vs Bad).

fig, ax = plt.subplots(figsize=(9, 5))

quality_counts = df['quality'].value_counts().sort_index()
colors = ['#4caf82' if q >= 7 else '#e05c5c' for q in quality_counts.index]

bars = ax.bar(quality_counts.index.astype(str), quality_counts.values,
              color=colors, edgecolor='none', width=0.6)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            str(int(bar.get_height())), ha='center', color='#888880', fontsize=9)

ax.set_title('🍷 Wine Quality Score Distribution', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Quality Score')
ax.set_ylabel('Count')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#4caf82', label='Good (score ≥ 7)'),
    Patch(facecolor='#e05c5c', label='Bad  (score < 7)'),
], facecolor='#1a1a1a', edgecolor='#2a2a2a', labelcolor='#e8e0d0')

plt.tight_layout()
plt.savefig('../images/quality_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

**Reading this chart:**
- Quality 5 and 6 dominate → severely imbalanced dataset
- Only ~20% of wines score ≥ 7 (Good)
- Predicting all 10 classes would be very difficult → we binarize

In [ ]:
# Correlation Heatmap
# INSIGHT: Look at the 'quality' row — alcohol is the strongest predictor.
# Volatile acidity is negatively correlated (more = worse wine).

corr = df.select_dtypes(include=[np.number]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 10))

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.4, linecolor='#1a1a1a',
            annot_kws={'size': 8, 'color': '#e8e0d0'}, ax=ax,
            cbar_kws={'shrink': 0.8})

ax.set_title('🌡️ Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=18)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('../images/heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

**Reading this heatmap:**
- `alcohol` ↑ → `quality` ↑ (strongest positive predictor)
- `volatile acidity` ↑ → `quality` ↓ (strongest negative predictor)
- `density` and `residual sugar` are highly correlated — multicollinearity risk
- `total sulfur dioxide` and `free sulfur dioxide` are also correlated (expected)

---
## 🏷️ 3. Feature Engineering — Binarize Target

In [ ]:
# Convert quality score → binary label
df['label'] = (df['quality'] >= 7).astype(int)

good = df['label'].sum()
bad  = len(df) - good

print(f'Good wines (1): {good} ({good/len(df)*100:.1f}%)')
print(f'Bad  wines (0): {bad}  ({bad/len(df)*100:.1f}%)')
df[['quality', 'label']].value_counts().sort_index()

---
## ✂️ 4. Train / Test Split + Feature Scaling

In [ ]:
# Separate features (X) from target (y)
drop_cols = ['quality', 'label'] + (['type'] if 'type' in df.columns else [])
X = df.drop(columns=drop_cols)
y = df['label']

feature_names = list(X.columns)
print(f'Features : {feature_names}')
print(f'X shape  : {X.shape}')
print(f'y shape  : {y.shape}')

In [ ]:
# Train/test split — stratify=y preserves Good/Bad ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

In [ ]:
# Feature scaling — fit ONLY on training data!
# If we fit on test data too → data leakage → inflated accuracy
scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)       # transform only, no fit

print('✅ Scaling complete!')
print(f'   X_train mean (first feature): {X_train_sc[:, 0].mean():.4f}  ← should be ≈0')
print(f'   X_train std  (first feature): {X_train_sc[:, 0].std():.4f}   ← should be ≈1')

---
## 🤖 5. Model Training

In [ ]:
# Train Logistic Regression (baseline)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

lr_acc = accuracy_score(y_test, lr.predict(X_test_sc))
print(f'Logistic Regression Accuracy: {lr_acc*100:.2f}%')

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)

rf_acc = accuracy_score(y_test, rf.predict(X_test_sc))
print(f'Random Forest Accuracy: {rf_acc*100:.2f}%')

---
## 📋 6. Model Evaluation

In [ ]:
# Compare models side by side
print('═' * 55)
print('  Model Comparison')
print('═' * 55)
print(f'  Logistic Regression : {lr_acc*100:.2f}%')
print(f'  Random Forest       : {rf_acc*100:.2f}%  ← winner')
print()

# Detailed report for best model
best_model = rf if rf_acc >= lr_acc else lr
y_pred     = best_model.predict(X_test_sc)

print('── Classification Report (Best Model) ────────')
print(classification_report(y_test, y_pred, target_names=['Bad (0)', 'Good (1)']))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 6))

cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Bad (0)', 'Good (1)'])
disp.plot(ax=ax, cmap='YlOrBr', colorbar=False, values_format='d')

for text in disp.text_.ravel():
    text.set_color('#0f0f0f')
    text.set_fontsize(18)
    text.set_fontweight('bold')

ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Predicted Label', color='#888880')
ax.set_ylabel('Actual Label',    color='#888880')

fig.text(0.5, 0.01, f'Overall Accuracy: {rf_acc*100:.2f}%',
         ha='center', color='#c8a96e', fontsize=12, fontweight='bold')

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('../images/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

**Reading the confusion matrix:**

|  | Predicted Bad | Predicted Good |
|---|---|---|
| **Actual Bad** | True Negative ✅ | False Positive ❌ |
| **Actual Good** | False Negative ❌ | True Positive ✅ |

- High diagonal values = model is correct most of the time
- More False Negatives than False Positives → model is conservative (tends to predict Bad)

In [ ]:
# Feature Importance (Random Forest only)
importances  = rf.feature_importances_
sorted_idx   = np.argsort(importances)
sorted_feats = [feature_names[i] for i in sorted_idx]
sorted_imps  = importances[sorted_idx]

colors = ['#c8a96e' if i >= len(sorted_idx)-3 else '#4a6fa5'
          for i in range(len(sorted_idx))]

fig, ax = plt.subplots(figsize=(10, 7))

bars = ax.barh(sorted_feats, sorted_imps, color=colors, edgecolor='none', height=0.65)

for bar, val in zip(bars, sorted_imps):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', color='#888880', fontsize=9)

ax.set_title('🌲 Random Forest — Feature Importance', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Importance Score')
ax.set_xlim(0, sorted_imps.max() * 1.18)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#c8a96e', label='Top 3 Features'),
    Patch(facecolor='#4a6fa5', label='Other Features'),
], facecolor='#1a1a1a', edgecolor='#2a2a2a', labelcolor='#e8e0d0')

plt.tight_layout()
plt.savefig('../images/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

**Reading feature importance:**
- `alcohol` is almost always the #1 predictor — higher alcohol = better wine
- `volatile acidity` ranks high because it negatively impacts taste
- Features near zero contribute almost nothing → candidates for removal

---
## 💾 7. Save Model & Scaler

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

# Save best model
joblib.dump(best_model, '../models/wine_quality_model.pkl')
print('✅ Model  saved → models/wine_quality_model.pkl')

# Save scaler (MUST save for consistent predictions later)
joblib.dump(scaler, '../models/scaler.pkl')
print('✅ Scaler saved → models/scaler.pkl')

---
## 🔮 8. Make a Prediction

In [ ]:
# Load and predict with a new wine sample
model  = joblib.load('../models/wine_quality_model.pkl')
scaler = joblib.load('../models/scaler.pkl')

new_wine = pd.DataFrame([{
    'fixed acidity'       : 7.4,
    'volatile acidity'    : 0.35,
    'citric acid'         : 0.40,
    'residual sugar'      : 2.1,
    'chlorides'           : 0.074,
    'free sulfur dioxide' : 28.0,
    'total sulfur dioxide': 92.0,
    'density'             : 0.9940,
    'pH'                  : 3.20,
    'sulphates'           : 0.73,
    'alcohol'             : 11.5
}])

new_wine_scaled = scaler.transform(new_wine)
pred            = model.predict(new_wine_scaled)[0]
proba           = model.predict_proba(new_wine_scaled)[0]

label_map = {0: '❌ Bad Quality', 1: '✅ Good Quality'}

print(f'Prediction  : {label_map[pred]}')
print(f'Confidence  : {proba[pred]*100:.1f}%')
print(f'Probability : Bad={proba[0]*100:.1f}%  Good={proba[1]*100:.1f}%')

---
## 📝 Summary

| Step | Action | Key Decision |
|---|---|---|
| 1 | Load data | Handle semicolon separator |
| 2 | EDA | Identified alcohol as top predictor |
| 3 | Feature engineering | Binary label (quality ≥ 7) |
| 4 | Split + scale | Stratified 80/20, StandardScaler |
| 5 | Train | LR (baseline) vs Random Forest |
| 6 | Evaluate | RF wins with ~88% accuracy |
| 7 | Save | joblib for model + scaler |
| 8 | Predict | Load → scale → predict |

### 🚀 Next Steps
- Try `GridSearchCV` to tune Random Forest hyperparameters
- Handle class imbalance with `SMOTE` or `class_weight='balanced'`
- Add cross-validation for more robust accuracy estimate
- Deploy the Streamlit app: `streamlit run app.py`